In [41]:
import rotation_functions as rot
import numpy as np

In [42]:
# HGA defaults pulled from Cycle 4 STOP Observatory Thermal Model
dish_pointing_0_0 = np.array([-0.17384746, -0.00824553, 0.98473807])
y_gimbal_rotation_axis = np.array([-0.00423894,	0.99996195,	0.00762465])
x_gimbal_rotation_axis = np.array([0.98476347,	0.00284872,	0.1738758])
HGA_initial_configuration = [dish_pointing_0_0, y_gimbal_rotation_axis, x_gimbal_rotation_axis]

# defaults for observatory rotations in FOR
y_obs_rotation_axis = np.array([0,	1,	0])
x_obs_rotation_axis = np.array([1,	0,	0])
sun_angle_0_0 =   np.array([0,	0,	1])
obs_initial = [sun_angle_0_0, y_obs_rotation_axis, x_obs_rotation_axis]

In [43]:
# Custom functions for our specific case, set defaults for RST
def rotate_HGA(HGA_inputs,HGA_initial_config = HGA_initial_configuration):
    y_track, x_track = HGA_inputs
    return rot.rotate_two_vectors(HGA_initial_config[0],y_track,x_track,HGA_initial_config[1],HGA_initial_config[2])

def rotate_HGA_coordinates_within_OBS(obs_FOR_attitude, HGA_point = HGA_initial_configuration,axis_1 = obs_initial[1],axis_2 = obs_initial[2]):
    y_obs,x_obs = obs_FOR_attitude
    sun_vector = rot.rotate_two_vectors(np.array([0,	0,	1]),y_obs,x_obs,axis_1,axis_2)
    
    HGA_rotated_with_obs = []
    for i in HGA_point:
        HGA_rotated_with_obs.append(rot.rotate_two_vectors(i,y_obs,x_obs,axis_1,axis_2))
    return HGA_rotated_with_obs, sun_vector


In [44]:
# input HGA configuration, assumes Y0 X0 but can input cust
def define_target(HGA_inputs,obs_FOR_attitude = [0,0]):
    y_HGA, x_HGA = HGA_inputs
    y_obs,x_obs = obs_FOR_attitude

    HGA_rotated_with_obs, v_solar = rotate_HGA_coordinates_within_OBS(obs_FOR_attitude)
    v_target = rotate_HGA(HGA_inputs,HGA_rotated_with_obs)
    
    print("Target for Y" + str(y_obs) + " X" + str(x_obs))
    print("With HG y_track = " + str(y_HGA) + " and HG x_track = " + str(x_HGA))
    print(v_target)
    print("")

    pointing_offset_angle = rot.angle_between_vectors(np.array([0,	0,	1]), v_target)
    print("Angle between HGA pointing & sun = " + str(pointing_offset_angle))
    print("")
    
    return v_target